In [1]:
import os
import sys
os.chdir("/workspaces/dev")
paths = [
    "./modules/python-utils",
    "./modules/ai-utils",
]
for path in paths:
    sys.path.append(os.path.abspath(path))
print(f"Current Python version: {sys.version}")

Current Python version: 3.10.12 (main, May 27 2025, 17:12:29) [GCC 11.4.0]


In [2]:
from faster_whisper import WhisperModel
from pathlib import Path
import librosa
import json

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from sj_ai_utils.asr.whisper_utils import *
from sj_ai_utils.datasets.libri_speech_asr_corpus import search_all_ref_and_hyp
from sj_ai_utils.evaluator.sclite_utils import *
from sj_utils.audio_utils import *
from sj_utils.string_utils import *
from sj_utils.collection_utils import SafetyDict

In [4]:
from rt_whisper import streamers
from rt_whisper.data import Param, Result

In [5]:
SOURCE = "/workspaces/dev/datasets/LibriSpeechASRcorpus/train"
MODEL_SIZE = "large-v3"
SAMPLE_RATE = 16000
OUTPUT = "/workspaces/dev/generated_datasets/"

In [6]:
whisper_model = WhisperModel(MODEL_SIZE, device="cuda", compute_type="float16")

In [7]:
hyperparameters = SafetyDict({
    "sentence_max_prev_sentence": 1,
    "weighted_and_offset_token_boundary": 8000,
    "duration_filter_z":{
        "default": 2.0,
        "ko": 2.0,
        "en": 3.0,
    },
    "probability_filter":{
        "z":{
            "default": 3.0,
            "ko": 3.0,
            "en": 3.0,
        },
        "min_prob": {
            "default": 1.0,
            "ko": 0.4,
            "en": 0.4
        },
    },
    "selector":{
        "search_range_sc": {
            "default": 24000,
            "ko": 24000,
            "en": 20000,
        },
        "threshold":{
            "default": 0.5,
            "ko": 0.25,
            "en": 0.25
        },
        "padding": {
            "default": 3200,
            "ko": 3200,
            "en": 3200
        },
        "tolerance": {
            "default": 8000,
            "ko": 8000,
            "en": 8000
        }
    },
    "max_overlap_duration": 96000
})

In [8]:
token_streamer = streamers.get_token_streamer_with_vad_v2(
    hyperparameter= hyperparameters
)

In [9]:
src = Path(SOURCE)
output = Path(OUTPUT)
output.mkdir(parents=True, exist_ok=True)

In [15]:
def transcriber(flac:Path) -> TRNFormat:
    audio, _ = librosa.load(flac, sr=SAMPLE_RATE)

    data = {
        "path": str(flac),
        "segment": []
    }
    param = Param()

    start = 0
    for segment in segment_audio(audio):
        param.chunk = segment
        param.language="en"
        ctx = token_streamer.process(param, get_context=True)
        result:Result = ctx.extract()
        param.update(result)
        length = len(segment)
        data["segment"].append({
            "start": start,
            "end": start + length,
            "tokens": [{
                "start": token.start,
                "end": token.end,
                "text": token.text,
                "probability": token.probability,
            } for token in ctx.segment_tokens if token.is_word]
        })
        start = start + length

    return data

In [16]:
audios = list(src.rglob("*.flac"))

In [18]:
for audio in audios[:1]:
    data = transcriber(audio)
    segments, _ = whisper_model.transcribe(
        audio,
        beam_size=5,
        language="en",
        word_timestamps=True,
        vad_filter=False
    )
    segments = list(segments)
    words = [{
        "start": w.start,
        "end": w.end,
        "text": w.word,
        # "probability": w.probability
    } for segment in segments for w in segment.words]

    with open(output / f"{audio.stem}.json", "w", encoding="utf-8") as f:
        json.dump({
            "X": data,
            "Y": words,
        }, f, ensure_ascii=False, indent=4)